# 04. 대전 전체 일별 물량 예측

## 현재 작업 범위

이 노트북은 하이브리드 투스테이지 모델링의 첫 작업 단위로 다음 항목만 수행한다.

1. Train/Test 데이터 로드 및 무결성 재검증
2. Stage 1 평시 모델의 입력 피처와 Target 확정
3. Train 내부 expanding-window 검증 Fold 구성
4. 평시 물량 단순 기준 모델 성능 측정
5. Stage 1 평시 ML 후보 비교 및 선택

> 2026년 Test는 구조만 확인하고 모델 선택이나 성능 비교에 사용하지 않는다. Stage 2 이벤트 효과 모델은 다음 작업 단위에서 진행한다.

## 1. 라이브러리 및 경로 설정

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error


RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

TRAIN_PATH = PROJECT_ROOT / "data" / "processed" / "train.csv"
TEST_PATH = PROJECT_ROOT / "data" / "processed" / "test.csv"

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

## 2. Train/Test 로드 및 무결성 검증

In [2]:
EXPECTED_COLUMNS = [
    "접수일자",
    "data_split",
    "접수지역",
    "접수통수",
    "요일",
    "제1기분 자동차세",
    "재산세(건축)",
    "정기분 주민세",
    "주민세(사업소분)",
    "재산세(토지)",
    "제2기분 자동차세",
    "사회보험료 통합",
    "event_count",
    "is_event",
    "month",
    "day",
    "weekday_월",
    "weekday_화",
    "weekday_수",
    "weekday_목",
    "weekday_금",
    "weekday_토",
    "weekday_일",
    "days_since_prev",
    "lag_1",
    "lag_5",
    "rolling_mean_5",
    "rolling_mean_20",
]

assert TRAIN_PATH.exists(), f"Train 파일이 없습니다: {TRAIN_PATH}"
assert TEST_PATH.exists(), f"Test 파일이 없습니다: {TEST_PATH}"

train = pd.read_csv(
    TRAIN_PATH,
    encoding="utf-8-sig",
    parse_dates=["접수일자"],
)
test = pd.read_csv(
    TEST_PATH,
    encoding="utf-8-sig",
    parse_dates=["접수일자"],
)

assert train.columns.tolist() == EXPECTED_COLUMNS
assert test.columns.tolist() == EXPECTED_COLUMNS
assert train.shape == (475, 28)
assert test.shape == (122, 28)
assert train.isna().sum().sum() == 0
assert test.isna().sum().sum() == 0
assert train["접수일자"].is_unique
assert test["접수일자"].is_unique
assert train["접수일자"].is_monotonic_increasing
assert test["접수일자"].is_monotonic_increasing
assert train["data_split"].eq("train").all()
assert test["data_split"].eq("test").all()
assert train["접수일자"].max() < test["접수일자"].min()
assert set(train["접수일자"]).isdisjoint(set(test["접수일자"]))
assert train["is_event"].value_counts().sort_index().to_dict() == {
    0: 334,
    1: 141,
}
assert test["is_event"].value_counts().sort_index().to_dict() == {
    0: 96,
    1: 26,
}

dataset_summary = pd.DataFrame(
    {
        "구분": ["Train", "Test"],
        "행 수": [len(train), len(test)],
        "열 수": [len(train.columns), len(test.columns)],
        "시작일": [
            train["접수일자"].min().date().isoformat(),
            test["접수일자"].min().date().isoformat(),
        ],
        "종료일": [
            train["접수일자"].max().date().isoformat(),
            test["접수일자"].max().date().isoformat(),
        ],
        "평시": [
            int(train["is_event"].eq(0).sum()),
            int(test["is_event"].eq(0).sum()),
        ],
        "이벤트": [
            int(train["is_event"].eq(1).sum()),
            int(test["is_event"].eq(1).sum()),
        ],
        "결측치": [
            int(train.isna().sum().sum()),
            int(test.isna().sum().sum()),
        ],
    }
)

display(dataset_summary)

,구분,행 수,열 수,시작일,종료일,평시,이벤트,결측치
0,Train,475,28,2024-01-30,2025-12-31,334,141,0
1,Test,122,28,2026-01-02,2026-06-30,96,26,0


## 3. Stage 1 Target 및 피처 확정

In [3]:
TARGET_COLUMN = "접수통수"

EVENT_COLUMNS = [
    "제1기분 자동차세",
    "재산세(건축)",
    "정기분 주민세",
    "주민세(사업소분)",
    "재산세(토지)",
    "제2기분 자동차세",
    "사회보험료 통합",
]

STAGE1_FEATURES = [
    "month",
    "day",
    "weekday_월",
    "weekday_화",
    "weekday_수",
    "weekday_목",
    "weekday_금",
    "weekday_토",
    "weekday_일",
    "days_since_prev",
    "lag_1",
    "lag_5",
    "rolling_mean_5",
    "rolling_mean_20",
]

EXCLUDED_MODEL_COLUMNS = [
    "접수일자",
    "data_split",
    "접수지역",
    TARGET_COLUMN,
    "요일",
    "event_count",
    "is_event",
    *EVENT_COLUMNS,
]

assert TARGET_COLUMN in train.columns
assert set(STAGE1_FEATURES).issubset(train.columns)
assert set(EXCLUDED_MODEL_COLUMNS).issubset(train.columns)
assert set(STAGE1_FEATURES).isdisjoint(EXCLUDED_MODEL_COLUMNS)
assert train[STAGE1_FEATURES].notna().all(axis=None)
assert test[STAGE1_FEATURES].notna().all(axis=None)
assert train[TARGET_COLUMN].gt(0).all()
assert test[TARGET_COLUMN].gt(0).all()

baseline_train_all = train.loc[train["is_event"].eq(0)].copy()
event_train_all = train.loc[train["is_event"].eq(1)].copy()

feature_definition = pd.DataFrame(
    {
        "항목": [
            "Target",
            "Stage 1 피처 수",
            "Stage 1 학습 가능 평시 행",
            "Stage 2 후보 이벤트 행",
            "Test 사용 여부(현재 단계)",
        ],
        "값": [
            TARGET_COLUMN,
            len(STAGE1_FEATURES),
            len(baseline_train_all),
            len(event_train_all),
            "구조 검증만 수행",
        ],
    }
)

display(feature_definition)
display(pd.DataFrame({"Stage 1 피처": STAGE1_FEATURES}))

,항목,값
0,Target,접수통수
1,Stage 1 피처 수,14
2,Stage 1 학습 가능 평시 행,334
3,Stage 2 후보 이벤트 행,141
4,Test 사용 여부(현재 단계),구조 검증만 수행


,Stage 1 피처
0,month
1,day
2,weekday_월
3,weekday_화
4,weekday_수
5,weekday_목
6,weekday_금
7,weekday_토
8,weekday_일
9,days_since_prev


## 4. Train 내부 expanding-window Fold 구성

In [4]:
FOLD_SPECS = [
    {
        "fold": "Fold 1",
        "train_start": "2024-01-30",
        "train_end": "2024-06-30",
        "valid_start": "2024-07-01",
        "valid_end": "2024-12-31",
    },
    {
        "fold": "Fold 2",
        "train_start": "2024-01-30",
        "train_end": "2024-12-31",
        "valid_start": "2025-01-01",
        "valid_end": "2025-06-30",
    },
    {
        "fold": "Fold 3",
        "train_start": "2024-01-30",
        "train_end": "2025-06-30",
        "valid_start": "2025-07-01",
        "valid_end": "2025-12-31",
    },
]

time_folds = {}
fold_summary_rows = []

for spec in FOLD_SPECS:
    fold_name = spec["fold"]
    train_start = pd.Timestamp(spec["train_start"])
    train_end = pd.Timestamp(spec["train_end"])
    valid_start = pd.Timestamp(spec["valid_start"])
    valid_end = pd.Timestamp(spec["valid_end"])

    fold_train_all = train.loc[
        train["접수일자"].between(train_start, train_end)
    ].copy()
    fold_valid_all = train.loc[
        train["접수일자"].between(valid_start, valid_end)
    ].copy()

    fold_train_baseline = fold_train_all.loc[
        fold_train_all["is_event"].eq(0)
    ].copy()
    fold_valid_baseline = fold_valid_all.loc[
        fold_valid_all["is_event"].eq(0)
    ].copy()

    assert not fold_train_baseline.empty
    assert not fold_valid_baseline.empty
    assert fold_train_baseline["접수일자"].max() < (
        fold_valid_baseline["접수일자"].min()
    )
    assert fold_train_baseline["접수일자"].max() < test["접수일자"].min()
    assert fold_valid_baseline["접수일자"].max() < test["접수일자"].min()

    time_folds[fold_name] = {
        "train_all": fold_train_all,
        "valid_all": fold_valid_all,
        "train_baseline": fold_train_baseline,
        "valid_baseline": fold_valid_baseline,
    }

    fold_summary_rows.append(
        {
            "Fold": fold_name,
            "학습 전체": len(fold_train_all),
            "학습 평시": len(fold_train_baseline),
            "학습 이벤트": int(fold_train_all["is_event"].eq(1).sum()),
            "검증 전체": len(fold_valid_all),
            "검증 평시": len(fold_valid_baseline),
            "검증 이벤트": int(fold_valid_all["is_event"].eq(1).sum()),
            "학습 종료일": (
                fold_train_all["접수일자"].max().date().isoformat()
            ),
            "검증 시작일": (
                fold_valid_all["접수일자"].min().date().isoformat()
            ),
            "검증 종료일": (
                fold_valid_all["접수일자"].max().date().isoformat()
            ),
        }
    )

fold_summary = pd.DataFrame(fold_summary_rows)
display(fold_summary)

,Fold,학습 전체,학습 평시,학습 이벤트,검증 전체,검증 평시,검증 이벤트,학습 종료일,검증 시작일,검증 종료일
0,Fold 1,104,82,22,124,80,44,2024-06-28,2024-07-01,2024-12-31
1,Fold 2,228,162,66,123,94,29,2024-12-31,2025-01-02,2025-06-30
2,Fold 3,351,256,95,124,78,46,2025-06-30,2025-07-01,2025-12-31


## 5. 회귀 평가 지표

In [5]:
def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    assert y_true.shape == y_pred.shape
    assert np.isfinite(y_true).all()
    assert np.isfinite(y_pred).all()
    assert (y_true > 0).all(), "MAPE 계산을 위해 실제값이 0보다 커야 합니다."

    absolute_error = np.abs(y_true - y_pred)

    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAPE(%)": np.mean(absolute_error / y_true) * 100,
        "WAPE(%)": absolute_error.sum() / np.abs(y_true).sum() * 100,
    }


metric_test = regression_metrics(
    y_true=np.array([100.0, 200.0]),
    y_pred=np.array([90.0, 220.0]),
)

assert np.isclose(metric_test["MAE"], 15.0)
assert np.isclose(metric_test["RMSE"], np.sqrt(250.0))
assert np.isclose(metric_test["MAPE(%)"], 10.0)
assert np.isclose(metric_test["WAPE(%)"], 10.0)

display(pd.DataFrame([metric_test], index=["지표 함수 자체 검증"]))

,MAE,RMSE,MAPE(%),WAPE(%)
지표 함수 자체 검증,15.000,15.811,10.000,10.000


## 6. 평시 단순 기준 모델 평가

다음 기준 모델을 Train 내부 평시 검증 구간에서 비교한다.

- 학습 평시 평균
- 학습 평시 중앙값
- 직전 관측일 물량(`lag_1`)
- 직전 5개 관측일 전 물량(`lag_5`)
- 직전 5개 관측일 평균(`rolling_mean_5`)
- 직전 20개 관측일 평균(`rolling_mean_20`)
- 학습 평시 요일별 평균

Lag·이동평균은 매일 실제 물량이 갱신되는 rolling 1-step 예측 기준이다.

In [6]:
def weekday_mean_prediction(train_frame, valid_frame):
    weekday_means = train_frame.groupby("요일")[TARGET_COLUMN].mean()
    fallback = train_frame[TARGET_COLUMN].median()
    return valid_frame["요일"].map(weekday_means).fillna(fallback).to_numpy()


naive_result_rows = []

for fold_name, fold_data in time_folds.items():
    fold_train = fold_data["train_baseline"]
    fold_valid = fold_data["valid_baseline"]
    y_valid = fold_valid[TARGET_COLUMN].to_numpy()

    predictions = {
        "train_mean": np.full(
            len(fold_valid),
            fold_train[TARGET_COLUMN].mean(),
        ),
        "train_median": np.full(
            len(fold_valid),
            fold_train[TARGET_COLUMN].median(),
        ),
        "lag_1": fold_valid["lag_1"].to_numpy(),
        "lag_5": fold_valid["lag_5"].to_numpy(),
        "rolling_mean_5": fold_valid["rolling_mean_5"].to_numpy(),
        "rolling_mean_20": fold_valid["rolling_mean_20"].to_numpy(),
        "weekday_mean": weekday_mean_prediction(
            fold_train,
            fold_valid,
        ),
    }

    for model_name, y_pred in predictions.items():
        assert len(y_pred) == len(y_valid)
        assert np.isfinite(y_pred).all()

        metrics = regression_metrics(y_valid, y_pred)
        naive_result_rows.append(
            {
                "Fold": fold_name,
                "모델": model_name,
                "검증 평시 행": len(fold_valid),
                **metrics,
            }
        )

naive_fold_results = pd.DataFrame(naive_result_rows)
naive_summary = (
    naive_fold_results.groupby("모델", as_index=False)
    .agg(
        Fold수=("Fold", "nunique"),
        평균_MAE=("MAE", "mean"),
        평균_RMSE=("RMSE", "mean"),
        평균_MAPE=("MAPE(%)", "mean"),
        평균_WAPE=("WAPE(%)", "mean"),
        MAE_표준편차=("MAE", "std"),
    )
    .sort_values(["평균_MAE", "평균_RMSE"])
    .reset_index(drop=True)
)

assert naive_fold_results.shape[0] == len(FOLD_SPECS) * 7
assert naive_summary["Fold수"].eq(3).all()

display(naive_fold_results)
display(naive_summary)

,Fold,모델,검증 평시 행,MAE,RMSE,MAPE(%),WAPE(%)
0,Fold 1,train_mean,80,"35,598.148","42,722.936",70.759,44.676
1,Fold 1,train_median,80,"33,904.900","40,461.359",61.908,42.551
2,Fold 1,lag_1,80,"45,960.875","56,626.282",73.484,57.682
3,Fold 1,lag_5,80,"52,290.238","105,019.598",85.612,65.625
4,Fold 1,rolling_mean_5,80,"43,042.355","55,563.351",78.247,54.019
5,Fold 1,rolling_mean_20,80,"33,499.026","42,210.965",65.593,42.042
6,Fold 1,weekday_mean,80,"29,963.441","38,010.063",54.290,37.605
7,Fold 2,train_mean,94,"42,253.446","64,101.991",72.012,49.781
8,Fold 2,train_median,94,"40,870.319","64,133.556",66.098,48.151
9,Fold 2,lag_1,94,"55,719.777","80,121.113",82.504,65.646


,모델,Fold수,평균_MAE,평균_RMSE,평균_MAPE,평균_WAPE,MAE_표준편차
0,weekday_mean,3,"31,872.351","45,141.790",60.405,41.929,"5,804.907"
1,train_median,3,"34,370.112","45,891.234",65.788,45.115,"6,280.537"
2,rolling_mean_20,3,"36,662.765","48,256.940",70.074,48.039,"8,598.670"
3,train_mean,3,"36,728.515","48,271.412",75.368,48.466,"5,055.433"
4,rolling_mean_5,3,"41,377.300","57,149.321",76.595,54.147,"8,239.767"
5,lag_1,3,"45,859.076","60,492.118",74.253,59.961,"9,911.992"
6,lag_5,3,"49,751.244","92,102.388",90.648,65.798,"4,882.888"


## 7. 첫 작업 단위 결과

In [7]:
best_naive_by_mae = naive_summary.iloc[0]

checkpoint_summary = pd.DataFrame(
    {
        "항목": [
            "Train/Test 무결성",
            "Stage 1 평시 Train",
            "Stage 1 피처 수",
            "Train 내부 Fold",
            "단순 기준 모델 수",
            "평균 MAE 기준 최우수 단순 모델",
            "2026년 Test 성능 확인",
        ],
        "결과": [
            "통과",
            f"{len(baseline_train_all):,}행",
            len(STAGE1_FEATURES),
            len(time_folds),
            naive_summary["모델"].nunique(),
            best_naive_by_mae["모델"],
            "수행하지 않음",
        ],
    }
)

display(checkpoint_summary)

print("첫 모델링 작업 단위 완료")
print("- 데이터 로드 및 무결성 검증: 완료")
print("- Stage 1 피처·Target 정의: 완료")
print("- Train 내부 시간순 Fold: 완료")
print("- 평시 단순 기준 모델 평가: 완료")
print("- ML 모델 및 Stage 2 학습: 수행하지 않음")
print("- 2026년 Test 모델 성능 확인: 수행하지 않음")

,항목,결과
0,Train/Test 무결성,통과
1,Stage 1 평시 Train,334행
2,Stage 1 피처 수,14
3,Train 내부 Fold,3
4,단순 기준 모델 수,7
5,평균 MAE 기준 최우수 단순 모델,weekday_mean
6,2026년 Test 성능 확인,수행하지 않음


첫 모델링 작업 단위 완료
- 데이터 로드 및 무결성 검증: 완료
- Stage 1 피처·Target 정의: 완료
- Train 내부 시간순 Fold: 완료
- 평시 단순 기준 모델 평가: 완료
- ML 모델 및 Stage 2 학습: 수행하지 않음
- 2026년 Test 모델 성능 확인: 수행하지 않음


## 8. Stage 1 평시 ML 후보 정의

추가 패키지 설치 없이 현재 scikit-learn 환경에서 다음 회귀 모델을 비교한다.

- Ridge: `alpha=1`, `alpha=10`
- Random Forest
- Extra Trees
- HistGradientBoosting

각 모델은 원본 Target과 `log1p` Target을 각각 비교한다. 모델 선택 기준은 Train 내부 3개 Fold의 평균 MAE이며, 평균 RMSE를 2차 기준으로 사용한다. 2026년 Test는 사용하지 않는다.

In [8]:
from time import perf_counter

from sklearn.ensemble import (
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def ridge_factory(alpha):
    return make_pipeline(
        StandardScaler(),
        Ridge(alpha=alpha),
    )


def random_forest_factory():
    return RandomForestRegressor(
        n_estimators=500,
        max_depth=8,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )


def extra_trees_factory():
    return ExtraTreesRegressor(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=2,
        max_features=0.8,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )


def hist_gradient_boosting_factory():
    return HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_iter=300,
        max_leaf_nodes=15,
        min_samples_leaf=10,
        l2_regularization=1.0,
        random_state=RANDOM_STATE,
    )


ML_MODEL_SPECS = [
    {
        "모델": "ridge_alpha_1",
        "Target변환": "raw",
        "factory": lambda: ridge_factory(alpha=1.0),
    },
    {
        "모델": "ridge_alpha_1",
        "Target변환": "log1p",
        "factory": lambda: ridge_factory(alpha=1.0),
    },
    {
        "모델": "ridge_alpha_10",
        "Target변환": "raw",
        "factory": lambda: ridge_factory(alpha=10.0),
    },
    {
        "모델": "ridge_alpha_10",
        "Target변환": "log1p",
        "factory": lambda: ridge_factory(alpha=10.0),
    },
    {
        "모델": "random_forest",
        "Target변환": "raw",
        "factory": random_forest_factory,
    },
    {
        "모델": "random_forest",
        "Target변환": "log1p",
        "factory": random_forest_factory,
    },
    {
        "모델": "extra_trees",
        "Target변환": "raw",
        "factory": extra_trees_factory,
    },
    {
        "모델": "extra_trees",
        "Target변환": "log1p",
        "factory": extra_trees_factory,
    },
    {
        "모델": "hist_gradient_boosting",
        "Target변환": "raw",
        "factory": hist_gradient_boosting_factory,
    },
    {
        "모델": "hist_gradient_boosting",
        "Target변환": "log1p",
        "factory": hist_gradient_boosting_factory,
    },
]

assert len(ML_MODEL_SPECS) == 10

model_spec_summary = pd.DataFrame(
    [
        {
            "모델": spec["모델"],
            "Target변환": spec["Target변환"],
        }
        for spec in ML_MODEL_SPECS
    ]
)

display(model_spec_summary)

,모델,Target변환
0,ridge_alpha_1,raw
1,ridge_alpha_1,log1p
2,ridge_alpha_10,raw
3,ridge_alpha_10,log1p
4,random_forest,raw
5,random_forest,log1p
6,extra_trees,raw
7,extra_trees,log1p
8,hist_gradient_boosting,raw
9,hist_gradient_boosting,log1p


## 9. Stage 1 평시 ML Fold 비교

각 Fold에서 이벤트가 없는 평시 행만 학습·검증한다. 로그 Target 모델은 `log1p`로 학습하고 `expm1`으로 원 단위에 복원한 뒤 지표를 계산한다. 예측값을 임의로 0 이상으로 자르지 않으며 음수 예측 건수를 별도로 기록한다.

In [9]:
def fit_predict_model(spec, x_train, y_train, x_valid):
    model = spec["factory"]()
    target_transform = spec["Target변환"]

    if target_transform == "log1p":
        fit_target = np.log1p(y_train)
    elif target_transform == "raw":
        fit_target = y_train
    else:
        raise ValueError(f"지원하지 않는 Target 변환: {target_transform}")

    started_at = perf_counter()
    model.fit(x_train, fit_target)
    prediction = model.predict(x_valid)
    elapsed_seconds = perf_counter() - started_at

    if target_transform == "log1p":
        prediction = np.expm1(prediction)

    return model, np.asarray(prediction, dtype=float), elapsed_seconds


stage1_ml_result_rows = []

for fold_name, fold_data in time_folds.items():
    fold_train = fold_data["train_baseline"]
    fold_valid = fold_data["valid_baseline"]

    x_train = fold_train[STAGE1_FEATURES]
    y_train = fold_train[TARGET_COLUMN].to_numpy()
    x_valid = fold_valid[STAGE1_FEATURES]
    y_valid = fold_valid[TARGET_COLUMN].to_numpy()

    for spec in ML_MODEL_SPECS:
        _, y_pred, elapsed_seconds = fit_predict_model(
            spec=spec,
            x_train=x_train,
            y_train=y_train,
            x_valid=x_valid,
        )

        assert y_pred.shape == y_valid.shape
        assert np.isfinite(y_pred).all()

        metrics = regression_metrics(y_valid, y_pred)
        stage1_ml_result_rows.append(
            {
                "Fold": fold_name,
                "모델": spec["모델"],
                "Target변환": spec["Target변환"],
                "학습 평시 행": len(fold_train),
                "검증 평시 행": len(fold_valid),
                "음수 예측": int((y_pred < 0).sum()),
                "학습·예측 초": elapsed_seconds,
                **metrics,
            }
        )

stage1_ml_fold_results = pd.DataFrame(stage1_ml_result_rows)
stage1_ml_summary = (
    stage1_ml_fold_results.groupby(
        ["모델", "Target변환"],
        as_index=False,
    )
    .agg(
        Fold수=("Fold", "nunique"),
        평균_MAE=("MAE", "mean"),
        평균_RMSE=("RMSE", "mean"),
        평균_MAPE=("MAPE(%)", "mean"),
        평균_WAPE=("WAPE(%)", "mean"),
        MAE_표준편차=("MAE", "std"),
        음수예측_합계=("음수 예측", "sum"),
        총실행초=("학습·예측 초", "sum"),
    )
    .sort_values(["평균_MAE", "평균_RMSE"])
    .reset_index(drop=True)
)

assert stage1_ml_fold_results.shape[0] == (
    len(FOLD_SPECS) * len(ML_MODEL_SPECS)
)
assert stage1_ml_summary["Fold수"].eq(3).all()

display(stage1_ml_fold_results)
display(stage1_ml_summary)

,Fold,모델,Target변환,학습 평시 행,검증 평시 행,음수 예측,학습·예측 초,MAE,RMSE,MAPE(%),WAPE(%)
0,Fold 1,ridge_alpha_1,raw,82,80,0,0.019,"28,420.371","40,290.117",40.791,35.668
1,Fold 1,ridge_alpha_1,log1p,82,80,0,0.002,"32,879.853","44,856.944",39.783,41.265
2,Fold 1,ridge_alpha_10,raw,82,80,0,0.002,"28,134.643","39,620.220",40.661,35.309
3,Fold 1,ridge_alpha_10,log1p,82,80,0,0.002,"32,292.900","43,979.773",39.625,40.528
4,Fold 1,random_forest,raw,82,80,0,0.337,"31,525.554","41,243.934",56.419,39.565
5,Fold 1,random_forest,log1p,82,80,0,0.357,"27,529.877","37,157.440",40.947,34.550
6,Fold 1,extra_trees,raw,82,80,0,0.202,"29,736.617","38,267.334",50.201,37.320
7,Fold 1,extra_trees,log1p,82,80,0,0.216,"28,591.106","38,138.327",41.212,35.882
8,Fold 1,hist_gradient_boosting,raw,82,80,1,0.244,"36,796.992","48,247.162",63.366,46.181
9,Fold 1,hist_gradient_boosting,log1p,82,80,0,0.210,"33,333.418","43,878.511",47.203,41.834


,모델,Target변환,Fold수,평균_MAE,평균_RMSE,평균_MAPE,평균_WAPE,MAE_표준편차,음수예측_합계,총실행초
0,extra_trees,log1p,3,"28,093.709","41,342.505",45.845,36.614,"6,848.386",0,0.613
1,random_forest,log1p,3,"28,459.691","41,624.004",47.319,37.196,"6,617.417",0,0.954
2,extra_trees,raw,3,"31,046.461","42,148.588",57.645,40.637,"6,897.641",0,0.600
3,ridge_alpha_10,log1p,3,"31,205.047","47,128.756",50.152,40.859,"6,066.961",0,0.005
4,ridge_alpha_1,log1p,3,"31,389.117","47,381.075",50.126,41.105,"6,008.767",0,0.006
5,ridge_alpha_10,raw,3,"31,608.579","46,392.581",57.361,41.743,"5,657.057",0,0.005
6,ridge_alpha_1,raw,3,"31,630.056","46,938.795",57.092,41.787,"5,364.287",0,0.024
7,random_forest,raw,3,"34,155.716","46,631.835",65.653,44.984,"6,188.071",0,0.925
8,hist_gradient_boosting,log1p,3,"34,701.343","48,843.184",57.246,45.563,"6,610.733",0,1.239
9,hist_gradient_boosting,raw,3,"38,351.721","52,445.438",70.055,50.148,"8,869.537",1,1.101


## 10. Stage 1 후보 선택

In [10]:
best_naive = naive_summary.iloc[0]
best_ml = stage1_ml_summary.iloc[0]

stage1_method_comparison = pd.DataFrame(
    [
        {
            "구분": "단순 기준",
            "후보": best_naive["모델"],
            "Target변환": "해당 없음",
            "평균_MAE": best_naive["평균_MAE"],
            "평균_RMSE": best_naive["평균_RMSE"],
            "평균_MAPE": best_naive["평균_MAPE"],
            "평균_WAPE": best_naive["평균_WAPE"],
        },
        {
            "구분": "ML",
            "후보": best_ml["모델"],
            "Target변환": best_ml["Target변환"],
            "평균_MAE": best_ml["평균_MAE"],
            "평균_RMSE": best_ml["평균_RMSE"],
            "평균_MAPE": best_ml["평균_MAPE"],
            "평균_WAPE": best_ml["평균_WAPE"],
        },
    ]
).sort_values(["평균_MAE", "평균_RMSE"]).reset_index(drop=True)

selected_stage1_candidate = stage1_method_comparison.iloc[0].to_dict()
ml_mae_improvement_vs_naive = (
    (best_naive["평균_MAE"] - best_ml["평균_MAE"])
    / best_naive["평균_MAE"]
    * 100
)

display(stage1_method_comparison)

print("Stage 1 평시 ML 비교 완료")
print(
    "- 최우수 단순 기준: "
    f"{best_naive['모델']} "
    f"(평균 MAE {best_naive['평균_MAE']:,.1f})"
)
print(
    "- 최우수 ML: "
    f"{best_ml['모델']} / {best_ml['Target변환']} "
    f"(평균 MAE {best_ml['평균_MAE']:,.1f})"
)
print(
    "- 최우수 ML의 단순 기준 대비 MAE 개선율: "
    f"{ml_mae_improvement_vs_naive:,.2f}%"
)
print(
    "- Stage 1 현재 선택 후보: "
    f"{selected_stage1_candidate['구분']} / "
    f"{selected_stage1_candidate['후보']} / "
    f"{selected_stage1_candidate['Target변환']}"
)
print("- 2026년 Test 성능 확인: 수행하지 않음")
print("- Stage 2 이벤트 효과 모델: 수행하지 않음")

,구분,후보,Target변환,평균_MAE,평균_RMSE,평균_MAPE,평균_WAPE
0,ML,extra_trees,log1p,"28,093.709","41,342.505",45.845,36.614
1,단순 기준,weekday_mean,해당 없음,"31,872.351","45,141.790",60.405,41.929


Stage 1 평시 ML 비교 완료
- 최우수 단순 기준: weekday_mean (평균 MAE 31,872.4)
- 최우수 ML: extra_trees / log1p (평균 MAE 28,093.7)
- 최우수 ML의 단순 기준 대비 MAE 개선율: 11.86%
- Stage 1 현재 선택 후보: ML / extra_trees / log1p
- 2026년 Test 성능 확인: 수행하지 않음
- Stage 2 이벤트 효과 모델: 수행하지 않음


## 11. 선택된 Stage 1 모델의 시간순 OOF 예측

Stage 2의 이벤트 효과를 계산할 때 실제 미래 정보를 사용하지 않도록, 각 검증 구간보다 앞선 평시 데이터만으로 Stage 1 모델을 다시 학습한다. 이렇게 만든 OOF(Out-of-Fold) 베이스라인 예측을 전체 검증일에 적용하고, 이벤트 날짜의 `실제 물량 - 베이스라인 예측 물량`을 Stage 2 학습 Target으로 사용한다.

- OOF 생성 범위: 2024년 하반기 ~ 2025년 하반기
- Stage 1 학습 데이터: 각 Fold 검증 시작일 이전의 평시 행만 사용
- 2024년 상반기: 최초 학습 구간이므로 OOF 잔차 학습 대상에서 제외
- 2026년 Test: 이 단계에서는 성능을 확인하지 않음

In [11]:
selected_stage1_spec = next(
    spec
    for spec in ML_MODEL_SPECS
    if (
        spec["모델"] == best_ml["모델"]
        and spec["Target변환"] == best_ml["Target변환"]
    )
)

stage1_oof_frames = []

for fold_name, fold_data in time_folds.items():
    fold_train = fold_data["train_baseline"]
    fold_valid = fold_data["valid_all"]

    _, baseline_prediction_raw, _ = fit_predict_model(
        spec=selected_stage1_spec,
        x_train=fold_train[STAGE1_FEATURES],
        y_train=fold_train[TARGET_COLUMN].to_numpy(),
        x_valid=fold_valid[STAGE1_FEATURES],
    )

    fold_oof = fold_valid.copy()
    fold_oof["Fold"] = fold_name
    fold_oof["stage1_baseline_prediction_raw"] = baseline_prediction_raw
    fold_oof["stage1_baseline_prediction"] = np.clip(
        baseline_prediction_raw,
        a_min=0,
        a_max=None,
    )
    fold_oof["event_residual"] = (
        fold_oof[TARGET_COLUMN]
        - fold_oof["stage1_baseline_prediction"]
    )
    stage1_oof_frames.append(fold_oof)

stage1_oof = (
    pd.concat(stage1_oof_frames, ignore_index=True)
    .sort_values("접수일자")
    .reset_index(drop=True)
)

expected_oof_rows = sum(
    len(fold_data["valid_all"])
    for fold_data in time_folds.values()
)

assert len(stage1_oof) == expected_oof_rows
assert stage1_oof["접수일자"].is_unique
assert stage1_oof["접수일자"].is_monotonic_increasing
assert stage1_oof["접수일자"].max() < test["접수일자"].min()
assert stage1_oof["stage1_baseline_prediction"].ge(0).all()
assert stage1_oof["event_residual"].notna().all()

stage1_oof_metric_rows = []
for segment_name, segment_mask in {
    "전체": pd.Series(True, index=stage1_oof.index),
    "평시": stage1_oof["is_event"].eq(0),
    "이벤트": stage1_oof["is_event"].eq(1),
}.items():
    segment = stage1_oof.loc[segment_mask]
    stage1_oof_metric_rows.append(
        {
            "구간": segment_name,
            "행 수": len(segment),
            **regression_metrics(
                segment[TARGET_COLUMN],
                segment["stage1_baseline_prediction"],
            ),
        }
    )

stage1_oof_metrics = pd.DataFrame(stage1_oof_metric_rows)
stage1_oof_summary = pd.DataFrame(
    {
        "항목": [
            "OOF 시작일",
            "OOF 종료일",
            "OOF 전체 행",
            "OOF 이벤트 행",
            "음수 원예측",
            "2026년 Test 성능 확인",
        ],
        "값": [
            stage1_oof["접수일자"].min().date().isoformat(),
            stage1_oof["접수일자"].max().date().isoformat(),
            len(stage1_oof),
            int(stage1_oof["is_event"].eq(1).sum()),
            int(
                stage1_oof["stage1_baseline_prediction_raw"].lt(0).sum()
            ),
            "수행하지 않음",
        ],
    }
)

display(stage1_oof_summary)
display(stage1_oof_metrics)

,항목,값
0,OOF 시작일,2024-07-01
1,OOF 종료일,2025-12-31
2,OOF 전체 행,371
3,OOF 이벤트 행,119
4,음수 원예측,0
5,2026년 Test 성능 확인,수행하지 않음


,구간,행 수,MAE,RMSE,MAPE(%),WAPE(%)
0,전체,371,"37,200.249","71,194.098",412.015,44.011
1,평시,252,"28,515.823","44,635.263",46.090,37.226
2,이벤트,119,"55,590.797","107,624.821","1,186.917",54.876


## 12. Stage 2 이벤트 잔차 모델의 시간순 검증

이벤트 행만 사용해 Stage 1 OOF 잔차를 학습한다. 검증 시점보다 앞서 생성된 OOF 잔차만 학습에 사용하며, 최종 비교는 잔차 자체가 아니라 `Stage 1 베이스라인 + Stage 2 조정값`의 실제 물량 예측 오차로 수행한다.

비교 후보는 Stage 1 단독, 전체 이벤트 잔차의 평균·중앙값, 이벤트 종류별 평균·중앙값, Ridge, Random Forest, Extra Trees, HistGradientBoosting이다. 잔차에는 음수가 존재할 수 있으므로 Stage 2 Target에는 로그 변환을 적용하지 않는다.

`2025-06-22`의 사회보험료 통합 이벤트 65통은 평가 지표를 왜곡하는 이상치로 간주하여 Stage 2 학습·검증에서만 제외한다. 원본 데이터와 Stage 1 학습 데이터는 변경하지 않는다.

In [12]:
STAGE2_FEATURES = [
    *EVENT_COLUMNS,
    "event_count",
    "month",
    "day",
    "weekday_월",
    "weekday_화",
    "weekday_수",
    "weekday_목",
    "weekday_금",
    "weekday_토",
    "weekday_일",
    "days_since_prev",
    "stage1_baseline_prediction",
]

STAGE2_OUTLIER_DATES = pd.to_datetime(["2025-06-22"])
stage2_excluded_event_rows = stage1_oof.loc[
    stage1_oof["접수일자"].isin(STAGE2_OUTLIER_DATES)
    & stage1_oof["is_event"].eq(1)
].copy()
stage2_event_oof = stage1_oof.loc[
    stage1_oof["is_event"].eq(1)
    & ~stage1_oof["접수일자"].isin(STAGE2_OUTLIER_DATES)
].copy()

assert len(stage2_excluded_event_rows) == 1
assert stage2_excluded_event_rows[TARGET_COLUMN].iloc[0] == 65
assert not stage2_event_oof["접수일자"].isin(STAGE2_OUTLIER_DATES).any()

STAGE2_FOLD_SPECS = [
    {
        "Fold": "Stage 2 Fold 1",
        "학습 OOF Fold": ["Fold 1"],
        "검증 OOF Fold": "Fold 2",
    },
    {
        "Fold": "Stage 2 Fold 2",
        "학습 OOF Fold": ["Fold 1", "Fold 2"],
        "검증 OOF Fold": "Fold 3",
    },
]

STAGE2_ML_SPECS = [
    {
        "후보": "ridge_alpha_1",
        "factory": lambda: ridge_factory(alpha=1.0),
    },
    {
        "후보": "ridge_alpha_10",
        "factory": lambda: ridge_factory(alpha=10.0),
    },
    {
        "후보": "random_forest",
        "factory": random_forest_factory,
    },
    {
        "후보": "extra_trees",
        "factory": extra_trees_factory,
    },
    {
        "후보": "hist_gradient_boosting",
        "factory": hist_gradient_boosting_factory,
    },
]


def event_stat_adjustment(train_frame, valid_frame, statistic):
    if statistic == "mean":
        stat_function = pd.Series.mean
    elif statistic == "median":
        stat_function = pd.Series.median
    else:
        raise ValueError(f"지원하지 않는 통계량: {statistic}")

    global_adjustment = float(
        stat_function(train_frame["event_residual"])
    )
    event_adjustments = {}
    for event_column in EVENT_COLUMNS:
        event_residuals = train_frame.loc[
            train_frame[event_column].eq(1),
            "event_residual",
        ]
        if not event_residuals.empty:
            event_adjustments[event_column] = float(
                stat_function(event_residuals)
            )

    predictions = []
    for _, row in valid_frame.iterrows():
        active_adjustments = [
            event_adjustments[event_column]
            for event_column in EVENT_COLUMNS
            if row[event_column] == 1
            and event_column in event_adjustments
        ]
        predictions.append(
            float(np.mean(active_adjustments))
            if active_adjustments
            else global_adjustment
        )

    return np.asarray(predictions, dtype=float)


stage2_result_rows = []

for fold_spec in STAGE2_FOLD_SPECS:
    stage2_train = stage2_event_oof.loc[
        stage2_event_oof["Fold"].isin(fold_spec["학습 OOF Fold"])
    ].copy()
    stage2_valid = stage2_event_oof.loc[
        stage2_event_oof["Fold"].eq(fold_spec["검증 OOF Fold"])
    ].copy()

    assert not stage2_train.empty
    assert not stage2_valid.empty
    assert stage2_train["접수일자"].max() < stage2_valid["접수일자"].min()

    candidate_adjustments = {
        "stage1_only": np.zeros(len(stage2_valid), dtype=float),
        "global_mean": np.repeat(
            stage2_train["event_residual"].mean(),
            len(stage2_valid),
        ),
        "global_median": np.repeat(
            stage2_train["event_residual"].median(),
            len(stage2_valid),
        ),
        "event_type_mean": event_stat_adjustment(
            stage2_train,
            stage2_valid,
            statistic="mean",
        ),
        "event_type_median": event_stat_adjustment(
            stage2_train,
            stage2_valid,
            statistic="median",
        ),
    }

    for model_spec in STAGE2_ML_SPECS:
        residual_model = model_spec["factory"]()
        residual_model.fit(
            stage2_train[STAGE2_FEATURES],
            stage2_train["event_residual"],
        )
        candidate_adjustments[model_spec["후보"]] = residual_model.predict(
            stage2_valid[STAGE2_FEATURES]
        )

    for candidate_name, adjustment in candidate_adjustments.items():
        adjustment = np.asarray(adjustment, dtype=float)
        final_prediction_raw = (
            stage2_valid["stage1_baseline_prediction"].to_numpy()
            + adjustment
        )
        final_prediction = np.clip(
            final_prediction_raw,
            a_min=0,
            a_max=None,
        )

        assert np.isfinite(adjustment).all()
        assert np.isfinite(final_prediction).all()

        stage2_result_rows.append(
            {
                "Fold": fold_spec["Fold"],
                "후보": candidate_name,
                "학습 이벤트 행": len(stage2_train),
                "검증 이벤트 행": len(stage2_valid),
                "평균 조정값": adjustment.mean(),
                "음수 최종 원예측": int((final_prediction_raw < 0).sum()),
                **regression_metrics(
                    stage2_valid[TARGET_COLUMN],
                    final_prediction,
                ),
            }
        )

stage2_fold_results = pd.DataFrame(stage2_result_rows)
stage2_summary = (
    stage2_fold_results.groupby("후보", as_index=False)
    .agg(
        Fold수=("Fold", "nunique"),
        평균_MAE=("MAE", "mean"),
        평균_RMSE=("RMSE", "mean"),
        평균_MAPE=("MAPE(%)", "mean"),
        평균_WAPE=("WAPE(%)", "mean"),
        MAE_표준편차=("MAE", "std"),
        음수최종원예측_합계=("음수 최종 원예측", "sum"),
    )
    .sort_values(["평균_MAE", "평균_RMSE"])
    .reset_index(drop=True)
)

assert TARGET_COLUMN not in STAGE2_FEATURES
assert set(STAGE2_FEATURES).issubset(stage2_event_oof.columns)
assert stage2_fold_results["후보"].nunique() == 10
assert stage2_summary["Fold수"].eq(2).all()

stage2_fold_definition = pd.DataFrame(
    [
        {
            "Fold": fold_spec["Fold"],
            "학습 OOF Fold": ", ".join(fold_spec["학습 OOF Fold"]),
            "검증 OOF Fold": fold_spec["검증 OOF Fold"],
        }
        for fold_spec in STAGE2_FOLD_SPECS
    ]
)

display(
    stage2_excluded_event_rows[
        ["접수일자", TARGET_COLUMN, "Fold", *EVENT_COLUMNS]
    ]
)
display(stage2_fold_definition)
display(pd.DataFrame({"Stage 2 ML 피처": STAGE2_FEATURES}))
display(stage2_fold_results)
display(stage2_summary)

,접수일자,접수통수,Fold,제1기분 자동차세,재산세(건축),정기분 주민세,주민세(사업소분),재산세(토지),제2기분 자동차세,사회보험료 통합
240,2025-06-22,65,Fold 2,0,0,0,0,0,0,1


,Fold,학습 OOF Fold,검증 OOF Fold
0,Stage 2 Fold 1,Fold 1,Fold 2
1,Stage 2 Fold 2,"Fold 1, Fold 2",Fold 3


,Stage 2 ML 피처
0,제1기분 자동차세
1,재산세(건축)
2,정기분 주민세
3,주민세(사업소분)
4,재산세(토지)
5,제2기분 자동차세
6,사회보험료 통합
7,event_count
8,month
9,day


,Fold,후보,학습 이벤트 행,검증 이벤트 행,평균 조정값,음수 최종 원예측,MAE,RMSE,MAPE(%),WAPE(%)
0,Stage 2 Fold 1,stage1_only,44,28,0.000,0,"50,844.400","102,065.255",46.713,53.477
1,Stage 2 Fold 1,global_mean,44,28,"37,432.508",0,"70,898.648","99,968.266",97.730,74.569
2,Stage 2 Fold 1,global_median,44,28,"-1,330.494",0,"50,548.066","102,391.156",45.449,53.165
3,Stage 2 Fold 1,event_type_mean,44,28,"7,295.107",0,"51,485.488","99,181.303",50.320,54.151
4,Stage 2 Fold 1,event_type_median,44,28,"-11,126.218",0,"48,386.599","104,641.543",35.353,50.892
5,Stage 2 Fold 1,ridge_alpha_1,44,28,"10,887.784",3,"74,276.362","118,732.361",85.618,78.122
6,Stage 2 Fold 1,ridge_alpha_10,44,28,"19,784.959",0,"63,210.724","104,718.195",69.796,66.483
7,Stage 2 Fold 1,random_forest,44,28,"6,864.872",0,"46,543.673","98,168.870",42.364,48.953
8,Stage 2 Fold 1,extra_trees,44,28,"12,732.689",0,"50,177.558","98,872.528",48.044,52.776
9,Stage 2 Fold 1,hist_gradient_boosting,44,28,"27,862.092",0,"59,288.727","103,971.810",71.769,62.358


,후보,Fold수,평균_MAE,평균_RMSE,평균_MAPE,평균_WAPE,MAE_표준편차,음수최종원예측_합계
0,event_type_median,2,"48,315.676","98,127.217",41.130,50.191,100.301,0
1,random_forest,2,"48,747.991","85,491.159",56.867,50.611,"3,117.377",0
2,global_median,2,"51,959.992","100,603.372",48.121,53.958,"1,996.765",0
3,stage1_only,2,"52,404.275","99,460.131",51.978,54.417,"2,205.996",0
4,event_type_mean,2,"53,448.892","91,061.449",64.253,55.497,"2,776.672",0
5,extra_trees,2,"53,455.732","87,753.521",65.277,55.487,"4,636.039",0
6,ridge_alpha_10,2,"60,721.094","94,277.232",77.358,63.109,"3,520.868",0
7,hist_gradient_boosting,2,"61,720.068","97,199.963",82.319,64.083,"3,438.436",1
8,ridge_alpha_1,2,"67,505.185","102,764.830",88.106,70.212,"9,575.890",3
9,global_mean,2,"67,915.142","96,629.423",99.324,70.589,"4,219.315",0


## 13. Stage 2 후보 선택

두 시간순 검증 Fold의 평균 MAE를 1순위, 평균 RMSE를 2순위로 사용한다. `stage1_only`도 같은 조건에서 비교하여, 이벤트 조정 모델이 실제로 Stage 1 단독보다 나아지는지 확인한다.

In [13]:
selected_stage2_candidate = stage2_summary.iloc[0].to_dict()
stage1_only_stage2_result = stage2_summary.loc[
    stage2_summary["후보"].eq("stage1_only")
].iloc[0]

stage2_mae_improvement_vs_stage1_only = (
    (
        stage1_only_stage2_result["평균_MAE"]
        - selected_stage2_candidate["평균_MAE"]
    )
    / stage1_only_stage2_result["평균_MAE"]
    * 100
)

stage2_selection_comparison = pd.DataFrame(
    [
        {
            "구분": "Stage 1 단독",
            "후보": "stage1_only",
            "평균_MAE": stage1_only_stage2_result["평균_MAE"],
            "평균_RMSE": stage1_only_stage2_result["평균_RMSE"],
            "평균_MAPE": stage1_only_stage2_result["평균_MAPE"],
            "평균_WAPE": stage1_only_stage2_result["평균_WAPE"],
        },
        {
            "구분": "선택 후보",
            "후보": selected_stage2_candidate["후보"],
            "평균_MAE": selected_stage2_candidate["평균_MAE"],
            "평균_RMSE": selected_stage2_candidate["평균_RMSE"],
            "평균_MAPE": selected_stage2_candidate["평균_MAPE"],
            "평균_WAPE": selected_stage2_candidate["평균_WAPE"],
        },
    ]
)

selected_candidate_name = selected_stage2_candidate["후보"]
stage2_foldwise_comparison = (
    stage2_fold_results.loc[
        stage2_fold_results["후보"].isin(
            ["stage1_only", selected_candidate_name]
        ),
        ["Fold", "후보", "MAE", "RMSE", "WAPE(%)"],
    ]
    .pivot(index="Fold", columns="후보")
)
stage2_foldwise_comparison.columns = [
    f"{metric}_{candidate}"
    for metric, candidate in stage2_foldwise_comparison.columns
]
stage2_foldwise_comparison = stage2_foldwise_comparison.reset_index()
stage2_foldwise_comparison["MAE_개선율(%)"] = (
    (
        stage2_foldwise_comparison["MAE_stage1_only"]
        - stage2_foldwise_comparison[
            f"MAE_{selected_candidate_name}"
        ]
    )
    / stage2_foldwise_comparison["MAE_stage1_only"]
    * 100
)

lowest_volume_event_rows = stage2_event_oof.nsmallest(
    5,
    TARGET_COLUMN,
)[["접수일자", TARGET_COLUMN, "Fold", *EVENT_COLUMNS]]
mape_is_distorted = bool(
    lowest_volume_event_rows[TARGET_COLUMN].min() < 1_000
)

display(stage2_selection_comparison)
display(stage2_foldwise_comparison)
display(lowest_volume_event_rows)

print("Stage 2 이벤트 잔차 모델 비교 완료")
print(
    "- Stage 1 OOF 이벤트 행: "
    f"{len(stage2_event_oof):,}행"
)
print(
    "- Stage 2 시간순 검증 Fold: "
    f"{len(STAGE2_FOLD_SPECS)}개"
)
print(
    "- 선택 후보: "
    f"{selected_stage2_candidate['후보']} "
    f"(평균 MAE {selected_stage2_candidate['평균_MAE']:,.1f})"
)
print(
    "- Stage 1 단독 대비 MAE 개선율: "
    f"{stage2_mae_improvement_vs_stage1_only:,.2f}%"
)
print(
    "- 선택 후보가 MAE를 개선한 검증 Fold: "
    f"{int(stage2_foldwise_comparison['MAE_개선율(%)'].gt(0).sum())}/"
    f"{len(stage2_foldwise_comparison)}개"
)
if mape_is_distorted:
    print(
        "- 주의: 실제 물량이 1,000통 미만인 이벤트 행 때문에 "
        "MAPE가 과도하게 커져 모델 선정에는 MAE와 RMSE를 사용함"
    )
print("- 2026년 Test 성능 확인: 수행하지 않음")

,구분,후보,평균_MAE,평균_RMSE,평균_MAPE,평균_WAPE
0,Stage 1 단독,stage1_only,"52,404.275","99,460.131",51.978,54.417
1,선택 후보,event_type_median,"48,315.676","98,127.217",41.130,50.191


,Fold,MAE_event_type_median,MAE_stage1_only,RMSE_event_type_median,RMSE_stage1_only,WAPE(%)_event_type_median,WAPE(%)_stage1_only,MAE_개선율(%)
0,Stage 2 Fold 1,"48,386.599","50,844.400","104,641.543","102,065.255",50.892,53.477,4.834
1,Stage 2 Fold 2,"48,244.752","53,964.150","91,612.892","96,855.007",49.490,55.358,10.599


,접수일자,접수통수,Fold,제1기분 자동차세,재산세(건축),정기분 주민세,주민세(사업소분),재산세(토지),제2기분 자동차세,사회보험료 통합
262,2025-07-22,20168,Fold 3,0,0,0,0,0,0,1
264,2025-07-24,20714,Fold 3,0,0,0,0,0,0,1
242,2025-06-24,21190,Fold 2,0,0,0,0,0,0,1
268,2025-07-30,21876,Fold 3,0,0,0,1,0,0,0
267,2025-07-29,23964,Fold 3,0,0,0,1,0,0,0


Stage 2 이벤트 잔차 모델 비교 완료
- Stage 1 OOF 이벤트 행: 118행
- Stage 2 시간순 검증 Fold: 2개
- 선택 후보: event_type_median (평균 MAE 48,315.7)
- Stage 1 단독 대비 MAE 개선율: 7.80%
- 선택 후보가 MAE를 개선한 검증 Fold: 2/2개
- 2026년 Test 성능 확인: 수행하지 않음


## 14. 최종 Stage 1 학습 및 Test 베이스라인 예측

앞에서 확정한 `Extra Trees + log1p` 모델을 2024~2025년 전체 평시 Train으로 학습하고, 2026년 Test 122일의 평시 기준 물량을 예측한다.

- Test 실제 물량은 입력 피처와 예측 결과 데이터에서 제외한다.
- 현재 피처의 lag·이동평균은 직전 관측일까지의 실제 물량을 사용하므로, 결과는 **관측일 단위 1-step-ahead 예측**으로 해석한다.
- 이 단계에서는 예측 개수·날짜 정렬·결측·음수·분포만 검증하며 Test 오차 지표는 계산하지 않는다.
- 이벤트 조정값은 다음 단계에서 별도로 산출한다.

In [14]:
FINAL_STAGE1_MODEL_NAME = selected_stage1_spec["모델"]
FINAL_STAGE1_TARGET_TRANSFORM = selected_stage1_spec["Target변환"]

x_final_stage1_train = baseline_train_all[STAGE1_FEATURES].copy()
y_final_stage1_train = baseline_train_all[TARGET_COLUMN].to_numpy()
x_test_stage1 = test[STAGE1_FEATURES].copy()

assert TARGET_COLUMN not in x_final_stage1_train.columns
assert TARGET_COLUMN not in x_test_stage1.columns
assert len(x_final_stage1_train) == 334
assert len(x_test_stage1) == 122
assert x_final_stage1_train.notna().all(axis=None)
assert x_test_stage1.notna().all(axis=None)

(
    final_stage1_model,
    final_stage1_prediction_raw,
    final_stage1_elapsed_seconds,
) = fit_predict_model(
    spec=selected_stage1_spec,
    x_train=x_final_stage1_train,
    y_train=y_final_stage1_train,
    x_valid=x_test_stage1,
)

final_stage1_prediction = np.clip(
    final_stage1_prediction_raw,
    a_min=0,
    a_max=None,
)

test_stage1_predictions = test[
    [
        "접수일자",
        "is_event",
        "event_count",
        *EVENT_COLUMNS,
    ]
].copy()
test_stage1_predictions["stage1_baseline_prediction_raw"] = (
    final_stage1_prediction_raw
)
test_stage1_predictions["stage1_baseline_prediction"] = (
    final_stage1_prediction
)

assert TARGET_COLUMN not in test_stage1_predictions.columns
assert len(test_stage1_predictions) == len(test)
assert test_stage1_predictions["접수일자"].equals(test["접수일자"])
assert test_stage1_predictions["접수일자"].is_unique
assert test_stage1_predictions["접수일자"].is_monotonic_increasing
assert np.isfinite(final_stage1_prediction_raw).all()
assert np.isfinite(final_stage1_prediction).all()
assert test_stage1_predictions["stage1_baseline_prediction"].ge(0).all()

final_stage1_training_summary = pd.DataFrame(
    {
        "항목": [
            "최종 모델",
            "Target 변환",
            "학습 평시 행",
            "학습 시작일",
            "학습 종료일",
            "Test 예측 행",
            "예측 방식",
            "Test 실제값 성능 비교",
        ],
        "값": [
            FINAL_STAGE1_MODEL_NAME,
            FINAL_STAGE1_TARGET_TRANSFORM,
            len(baseline_train_all),
            baseline_train_all["접수일자"].min().date().isoformat(),
            baseline_train_all["접수일자"].max().date().isoformat(),
            len(test_stage1_predictions),
            "관측일 단위 1-step-ahead",
            "수행하지 않음",
        ],
    }
)

prediction_distribution = pd.DataFrame(
    {
        "통계": [
            "최솟값",
            "25%",
            "중앙값",
            "평균",
            "75%",
            "최댓값",
        ],
        "Stage 1 베이스라인 예측": [
            np.min(final_stage1_prediction),
            np.quantile(final_stage1_prediction, 0.25),
            np.median(final_stage1_prediction),
            np.mean(final_stage1_prediction),
            np.quantile(final_stage1_prediction, 0.75),
            np.max(final_stage1_prediction),
        ],
    }
)

prediction_integrity_summary = pd.DataFrame(
    {
        "점검 항목": [
            "결측 예측",
            "비유한 예측",
            "음수 원예측",
            "0으로 보정된 예측",
            "날짜 중복",
            "날짜 정렬",
        ],
        "결과": [
            int(pd.isna(final_stage1_prediction).sum()),
            int((~np.isfinite(final_stage1_prediction)).sum()),
            int((final_stage1_prediction_raw < 0).sum()),
            int((final_stage1_prediction == 0).sum()),
            int(test_stage1_predictions["접수일자"].duplicated().sum()),
            bool(
                test_stage1_predictions["접수일자"].is_monotonic_increasing
            ),
        ],
    }
)

display(final_stage1_training_summary)
display(prediction_distribution)
display(prediction_integrity_summary)
display(
    test_stage1_predictions[
        [
            "접수일자",
            "is_event",
            "stage1_baseline_prediction",
        ]
    ].head()
)

print("최종 Stage 1 학습 및 Test 베이스라인 예측 완료")
print(
    "- 모델: "
    f"{FINAL_STAGE1_MODEL_NAME} / {FINAL_STAGE1_TARGET_TRANSFORM}"
)
print(f"- 전체 평시 Train: {len(baseline_train_all):,}행")
print(f"- Test 베이스라인 예측: {len(test_stage1_predictions):,}행")
print(
    "- 원예측 음수·결측: "
    f"{int((final_stage1_prediction_raw < 0).sum())}개 / "
    f"{int(pd.isna(final_stage1_prediction_raw).sum())}개"
)
print("- Test 실제 물량과의 성능 비교: 수행하지 않음")

,항목,값
0,최종 모델,extra_trees
1,Target 변환,log1p
2,학습 평시 행,334
3,학습 시작일,2024-01-30
4,학습 종료일,2025-12-31
5,Test 예측 행,122
6,예측 방식,관측일 단위 1-step-ahead
7,Test 실제값 성능 비교,수행하지 않음


,통계,Stage 1 베이스라인 예측
0,최솟값,"31,230.475"
1,25%,"53,263.385"
2,중앙값,"63,308.976"
3,평균,"71,360.755"
4,75%,"80,487.175"
5,최댓값,"146,101.600"


,점검 항목,결과
0,결측 예측,0
1,비유한 예측,0
2,음수 원예측,0
3,0으로 보정된 예측,0
4,날짜 중복,0
5,날짜 정렬,True


,접수일자,is_event,stage1_baseline_prediction
0,2026-01-02,0,"81,633.355"
1,2026-01-05,0,"131,335.133"
2,2026-01-06,0,"51,103.825"
3,2026-01-07,0,"56,327.558"
4,2026-01-08,0,"52,196.867"


최종 Stage 1 학습 및 Test 베이스라인 예측 완료
- 모델: extra_trees / log1p
- 전체 평시 Train: 334행
- Test 베이스라인 예측: 122행
- 원예측 음수·결측: 0개 / 0개
- Test 실제 물량과의 성능 비교: 수행하지 않음


## 15. 전체 Train 기반 Stage 2 이벤트 조정값 산출

최종 Stage 1 모델로 전체 Train 이벤트 날짜의 베이스라인 물량을 예측하고, `실제 물량 - Stage 1 베이스라인 예측` 잔차를 계산한다. 시간순 검증에서 선택된 방식에 따라 이벤트 종류별 잔차 중앙값을 최종 Stage 2 조정값으로 사용한다.

- 학습 대상: 2024~2025년 Train 이벤트 140일 (`2025-06-22` 65통 이상치 제외)
- 조정값: 이벤트 종류별 **부호가 있는 잔차 중앙값**
- 음수 조정값: 해당 이벤트 라벨 기간의 중앙적인 물량이 Stage 1 기준보다 낮았다는 의미이며 임의로 0으로 바꾸지 않는다.
- 미학습 이벤트: 전체 이벤트 잔차 중앙값을 fallback으로 사용한다.
- 복수 이벤트: 다음 결합 단계에서 활성 이벤트 조정값의 평균을 사용한다.
- 이 단계에서는 조정값을 Test 예측에 적용하거나 Test 성능을 계산하지 않는다.

In [15]:
def predict_with_fitted_stage1_model(model, target_transform, features):
    prediction = np.asarray(model.predict(features), dtype=float)
    if target_transform == "log1p":
        prediction = np.expm1(prediction)
    elif target_transform != "raw":
        raise ValueError(
            f"지원하지 않는 Target 변환: {target_transform}"
        )
    return prediction


final_stage2_train = event_train_all.loc[
    ~event_train_all["접수일자"].isin(STAGE2_OUTLIER_DATES)
].copy()
final_stage2_baseline_prediction_raw = (
    predict_with_fitted_stage1_model(
        model=final_stage1_model,
        target_transform=FINAL_STAGE1_TARGET_TRANSFORM,
        features=final_stage2_train[STAGE1_FEATURES],
    )
)
final_stage2_train["stage1_baseline_prediction"] = np.clip(
    final_stage2_baseline_prediction_raw,
    a_min=0,
    a_max=None,
)
final_stage2_train["event_residual"] = (
    final_stage2_train[TARGET_COLUMN]
    - final_stage2_train["stage1_baseline_prediction"]
)

assert len(final_stage2_train) == 140
assert not final_stage2_train["접수일자"].isin(STAGE2_OUTLIER_DATES).any()
assert np.isfinite(final_stage2_baseline_prediction_raw).all()
assert final_stage2_train["stage1_baseline_prediction"].ge(0).all()
assert final_stage2_train["event_residual"].notna().all()

FINAL_STAGE2_GLOBAL_FALLBACK = float(
    final_stage2_train["event_residual"].median()
)

final_adjustment_rows = []
for event_column in EVENT_COLUMNS:
    event_history = final_stage2_train.loc[
        final_stage2_train[event_column].eq(1)
    ].copy()
    oof_event_history = stage2_event_oof.loc[
        stage2_event_oof[event_column].eq(1)
    ].copy()

    assert not event_history.empty

    residuals = event_history["event_residual"]
    final_adjustment_rows.append(
        {
            "이벤트": event_column,
            "전체 Train 행": len(event_history),
            "학습 시작일": (
                event_history["접수일자"].min().date().isoformat()
            ),
            "학습 종료일": (
                event_history["접수일자"].max().date().isoformat()
            ),
            "최종 조정값(중앙값)": float(residuals.median()),
            "잔차 평균": float(residuals.mean()),
            "잔차 25%": float(residuals.quantile(0.25)),
            "잔차 75%": float(residuals.quantile(0.75)),
            "양수 잔차 비율(%)": float(residuals.gt(0).mean() * 100),
            "OOF 행": len(oof_event_history),
            "OOF 잔차 중앙값": (
                float(oof_event_history["event_residual"].median())
                if not oof_event_history.empty
                else np.nan
            ),
            "2026 Test 이벤트 행": int(test[event_column].sum()),
        }
    )

final_stage2_adjustment_table = pd.DataFrame(final_adjustment_rows)
FINAL_STAGE2_EVENT_ADJUSTMENTS = dict(
    zip(
        final_stage2_adjustment_table["이벤트"],
        final_stage2_adjustment_table["최종 조정값(중앙값)"],
    )
)

test_active_event_types = [
    event_column
    for event_column in EVENT_COLUMNS
    if test[event_column].sum() > 0
]
test_event_types_without_history = [
    event_column
    for event_column in test_active_event_types
    if event_column not in FINAL_STAGE2_EVENT_ADJUSTMENTS
]

assert not test_event_types_without_history
assert set(FINAL_STAGE2_EVENT_ADJUSTMENTS) == set(EVENT_COLUMNS)
assert np.isfinite(
    list(FINAL_STAGE2_EVENT_ADJUSTMENTS.values())
).all()
assert np.isfinite(FINAL_STAGE2_GLOBAL_FALLBACK)
assert "stage2_adjustment" not in test_stage1_predictions.columns

stage2_adjustment_summary = pd.DataFrame(
    {
        "항목": [
            "전체 Train 이벤트 행",
            "이벤트 플래그 합계",
            "복수 이벤트 행",
            "이벤트 종류",
            "전체 fallback 중앙값",
            "2026 Test 활성 이벤트 종류",
            "학습 이력 없는 Test 이벤트",
            "Test 적용 여부",
            "Test 성능 계산 여부",
        ],
        "값": [
            len(final_stage2_train),
            int(final_stage2_train[EVENT_COLUMNS].sum().sum()),
            int(final_stage2_train["event_count"].gt(1).sum()),
            len(EVENT_COLUMNS),
            FINAL_STAGE2_GLOBAL_FALLBACK,
            ", ".join(test_active_event_types),
            (
                ", ".join(test_event_types_without_history)
                if test_event_types_without_history
                else "없음"
            ),
            "아직 적용하지 않음",
            "수행하지 않음",
        ],
    }
)

test_active_adjustment_table = final_stage2_adjustment_table.loc[
    final_stage2_adjustment_table["이벤트"].isin(
        test_active_event_types
    ),
    [
        "이벤트",
        "전체 Train 행",
        "최종 조정값(중앙값)",
        "OOF 잔차 중앙값",
        "2026 Test 이벤트 행",
    ],
].reset_index(drop=True)

display(stage2_adjustment_summary)
display(final_stage2_adjustment_table)
display(test_active_adjustment_table)

print("전체 Train 기반 Stage 2 이벤트 조정값 산출 완료")
print(f"- Train 이벤트 행: {len(final_stage2_train):,}행")
print(
    "- 전체 fallback 잔차 중앙값: "
    f"{FINAL_STAGE2_GLOBAL_FALLBACK:,.1f}통"
)
print(
    "- 2026 Test 활성 이벤트: "
    f"{', '.join(test_active_event_types)}"
)
print("- 학습 이력 없는 Test 이벤트: 없음")
print("- Test 예측 결합 및 성능 계산: 수행하지 않음")

,항목,값
0,전체 Train 이벤트 행,140
1,이벤트 플래그 합계,142
2,복수 이벤트 행,2
3,이벤트 종류,7
4,전체 fallback 중앙값,"-1,752.755"
5,2026 Test 활성 이벤트 종류,"제1기분 자동차세, 사회보험료 통합"
6,학습 이력 없는 Test 이벤트,없음
7,Test 적용 여부,아직 적용하지 않음
8,Test 성능 계산 여부,수행하지 않음


,이벤트,전체 Train 행,학습 시작일,학습 종료일,최종 조정값(중앙값),잔차 평균,잔차 25%,잔차 75%,양수 잔차 비율(%),OOF 행,OOF 잔차 중앙값,2026 Test 이벤트 행
0,제1기분 자동차세,10,2024-06-10,2025-06-13,"18,263.875","61,996.398","-1,827.923","121,983.838",60.000,5,"13,086.766",5
1,재산세(건축),15,2024-07-09,2025-07-15,"18,201.786","90,342.173","-7,786.459","81,034.666",60.000,15,"14,892.491",0
2,정기분 주민세,10,2024-08-08,2025-08-14,"40,526.962","117,727.189","34,938.580","142,649.327",90.000,10,"25,781.731",0
3,주민세(사업소분),10,2024-07-25,2025-07-31,"-16,272.220","-12,820.186","-23,599.732","-9,127.799",20.000,10,"-24,330.933",0
4,재산세(토지),5,2025-09-09,2025-09-15,"67,473.636","106,062.275","-11,345.818","234,723.769",60.000,5,"63,989.368",0
5,제2기분 자동차세,10,2024-12-09,2025-12-15,"39,357.421","45,142.396","-14,406.257","53,182.438",70.000,10,"28,805.262",0
6,사회보험료 통합,82,2024-02-21,2025-12-24,"-6,571.633","4,910.431","-16,837.901","8,699.829",39.024,65,"-13,640.809",21


,이벤트,전체 Train 행,최종 조정값(중앙값),OOF 잔차 중앙값,2026 Test 이벤트 행
0,제1기분 자동차세,10,"18,263.875","13,086.766",5
1,사회보험료 통합,82,"-6,571.633","-13,640.809",21


전체 Train 기반 Stage 2 이벤트 조정값 산출 완료
- Train 이벤트 행: 140행
- 전체 fallback 잔차 중앙값: -1,752.8통
- 2026 Test 활성 이벤트: 제1기분 자동차세, 사회보험료 통합
- 학습 이력 없는 Test 이벤트: 없음
- Test 예측 결합 및 성능 계산: 수행하지 않음


## 16. Stage 1·2 예측 결합 및 날짜별 적용 검증

Stage 1 베이스라인 예측에 Stage 2 이벤트 종류별 중앙값 조정을 결합한다. 평시에는 조정값 0을 적용하고, 이벤트 날짜에는 활성 이벤트의 조정값을 적용한다.

- 평시: `최종 예측 = Stage 1 베이스라인`
- 단일 이벤트: `최종 예측 = Stage 1 베이스라인 + 해당 이벤트 중앙값`
- 복수 이벤트: 활성 이벤트 중앙값의 평균을 적용
- 미학습 이벤트: 전체 이벤트 잔차 중앙값 fallback 적용
- 최종 원예측이 음수이면 0으로 보정
- 이 단계에서는 날짜별 적용 규칙만 검증하며 Test 실제값과의 성능은 계산하지 않는다.

In [16]:
def resolve_stage2_adjustment(row):
    active_event_types = [
        event_column
        for event_column in EVENT_COLUMNS
        if row[event_column] == 1
    ]

    if not active_event_types:
        return pd.Series(
            {
                "active_event_types": "없음",
                "stage2_active_event_count": 0,
                "stage2_adjustment_source": "none",
                "stage2_adjustment": 0.0,
            }
        )

    event_adjustments = [
        FINAL_STAGE2_EVENT_ADJUSTMENTS.get(
            event_column,
            FINAL_STAGE2_GLOBAL_FALLBACK,
        )
        for event_column in active_event_types
    ]
    all_events_have_history = all(
        event_column in FINAL_STAGE2_EVENT_ADJUSTMENTS
        for event_column in active_event_types
    )

    return pd.Series(
        {
            "active_event_types": ", ".join(active_event_types),
            "stage2_active_event_count": len(active_event_types),
            "stage2_adjustment_source": (
                "event_type_median"
                if all_events_have_history
                else "global_fallback_median"
            ),
            "stage2_adjustment": float(np.mean(event_adjustments)),
        }
    )


test_hybrid_predictions = test_stage1_predictions.copy()
stage2_application = test_hybrid_predictions.apply(
    resolve_stage2_adjustment,
    axis=1,
)
test_hybrid_predictions = pd.concat(
    [test_hybrid_predictions, stage2_application],
    axis=1,
)
test_hybrid_predictions["hybrid_prediction_raw"] = (
    test_hybrid_predictions["stage1_baseline_prediction"]
    + test_hybrid_predictions["stage2_adjustment"]
)
test_hybrid_predictions["hybrid_prediction"] = np.clip(
    test_hybrid_predictions["hybrid_prediction_raw"],
    a_min=0,
    a_max=None,
)

baseline_test_mask = test_hybrid_predictions["is_event"].eq(0)
event_test_mask = test_hybrid_predictions["is_event"].eq(1)

assert TARGET_COLUMN not in test_hybrid_predictions.columns
assert len(test_hybrid_predictions) == len(test) == 122
assert int(baseline_test_mask.sum()) == 96
assert int(event_test_mask.sum()) == 26
assert not (baseline_test_mask & event_test_mask).any()
assert (baseline_test_mask | event_test_mask).all()
assert test_hybrid_predictions.loc[
    baseline_test_mask,
    "stage2_adjustment",
].eq(0).all()
assert np.allclose(
    test_hybrid_predictions.loc[
        baseline_test_mask,
        "hybrid_prediction",
    ],
    test_hybrid_predictions.loc[
        baseline_test_mask,
        "stage1_baseline_prediction",
    ],
)
assert test_hybrid_predictions.loc[
    event_test_mask,
    "stage2_adjustment_source",
].eq("event_type_median").all()
assert test_hybrid_predictions[
    "stage2_active_event_count"
].eq(test_hybrid_predictions["event_count"]).all()
assert np.isfinite(
    test_hybrid_predictions[
        ["stage2_adjustment", "hybrid_prediction"]
    ]
).all(axis=None)
assert test_hybrid_predictions["hybrid_prediction"].ge(0).all()

application_integrity_summary = pd.DataFrame(
    {
        "점검 항목": [
            "전체 Test 행",
            "평시 행",
            "이벤트 행",
            "조정된 평시 행",
            "조정된 이벤트 행",
            "fallback 적용 행",
            "복수 이벤트 행",
            "음수 최종 원예측",
            "0으로 보정된 최종 예측",
            "결측 최종 예측",
            "Test 성능 계산 여부",
        ],
        "결과": [
            len(test_hybrid_predictions),
            int(baseline_test_mask.sum()),
            int(event_test_mask.sum()),
            int(
                test_hybrid_predictions.loc[
                    baseline_test_mask,
                    "stage2_adjustment",
                ].ne(0).sum()
            ),
            int(
                test_hybrid_predictions.loc[
                    event_test_mask,
                    "stage2_adjustment",
                ].ne(0).sum()
            ),
            int(
                test_hybrid_predictions[
                    "stage2_adjustment_source"
                ].eq("global_fallback_median").sum()
            ),
            int(
                test_hybrid_predictions[
                    "stage2_active_event_count"
                ].gt(1).sum()
            ),
            int(
                test_hybrid_predictions[
                    "hybrid_prediction_raw"
                ].lt(0).sum()
            ),
            int(
                test_hybrid_predictions[
                    "hybrid_prediction"
                ].eq(0).sum()
            ),
            int(
                test_hybrid_predictions[
                    "hybrid_prediction"
                ].isna().sum()
            ),
            "수행하지 않음",
        ],
    }
)

segment_application_summary = (
    test_hybrid_predictions.assign(
        구간=np.where(event_test_mask, "이벤트", "평시")
    )
    .groupby("구간", as_index=False)
    .agg(
        행_수=("접수일자", "size"),
        평균_Stage1=("stage1_baseline_prediction", "mean"),
        평균_Stage2_조정=("stage2_adjustment", "mean"),
        최소_Stage2_조정=("stage2_adjustment", "min"),
        최대_Stage2_조정=("stage2_adjustment", "max"),
        평균_최종예측=("hybrid_prediction", "mean"),
    )
)

event_type_application_summary = (
    test_hybrid_predictions.loc[event_test_mask]
    .groupby("active_event_types", as_index=False)
    .agg(
        적용_일수=("접수일자", "size"),
        조정값=("stage2_adjustment", "first"),
        평균_Stage1=("stage1_baseline_prediction", "mean"),
        평균_최종예측=("hybrid_prediction", "mean"),
    )
)

event_application_detail = test_hybrid_predictions.loc[
    event_test_mask,
    [
        "접수일자",
        "active_event_types",
        "stage1_baseline_prediction",
        "stage2_adjustment",
        "hybrid_prediction",
    ],
].reset_index(drop=True)

display(application_integrity_summary)
display(segment_application_summary)
display(event_type_application_summary)
display(event_application_detail)

print("Stage 1·2 예측 결합 및 날짜별 적용 검증 완료")
print("- 평시 96일: Stage 2 조정 없이 Stage 1 유지")
print("- 이벤트 26일: 이벤트 종류별 중앙값 조정 적용")
print("- fallback·복수 이벤트 적용: 0일 / 0일")
print(
    "- 이벤트 날짜 평균 조정: "
    f"{test_hybrid_predictions.loc[event_test_mask, 'stage2_adjustment'].mean():,.1f}통"
)
print("- Test 실제 물량과의 성능 비교: 수행하지 않음")

,점검 항목,결과
0,전체 Test 행,122
1,평시 행,96
2,이벤트 행,26
3,조정된 평시 행,0
4,조정된 이벤트 행,26
5,fallback 적용 행,0
6,복수 이벤트 행,0
7,음수 최종 원예측,0
8,0으로 보정된 최종 예측,0
9,결측 최종 예측,0


,구간,행_수,평균_Stage1,평균_Stage2_조정,최소_Stage2_조정,최대_Stage2_조정,평균_최종예측
0,이벤트,26,"64,151.228","-1,795.574","-6,571.633","18,263.875","62,355.654"
1,평시,96,"73,313.336",0.000,0.000,0.000,"73,313.336"


,active_event_types,적용_일수,조정값,평균_Stage1,평균_최종예측
0,사회보험료 통합,21,"-6,571.633","61,147.966","54,576.333"
1,제1기분 자동차세,5,"18,263.875","76,764.930","95,028.804"


,접수일자,active_event_types,stage1_baseline_prediction,stage2_adjustment,hybrid_prediction
0,2026-01-21,사회보험료 통합,"54,582.755","-6,571.633","48,011.122"
1,2026-01-22,사회보험료 통합,"67,114.559","-6,571.633","60,542.927"
2,2026-01-23,사회보험료 통합,"61,128.892","-6,571.633","54,557.259"
3,2026-02-23,사회보험료 통합,"104,224.463","-6,571.633","97,652.830"
4,2026-02-24,사회보험료 통합,"52,780.488","-6,571.633","46,208.855"
5,2026-02-25,사회보험료 통합,"50,978.980","-6,571.633","44,407.347"
6,2026-03-23,사회보험료 통합,"95,187.789","-6,571.633","88,616.157"
7,2026-03-24,사회보험료 통합,"51,957.677","-6,571.633","45,386.044"
8,2026-03-25,사회보험료 통합,"60,704.223","-6,571.633","54,132.590"
9,2026-04-21,사회보험료 통합,"50,641.758","-6,571.633","44,070.125"


Stage 1·2 예측 결합 및 날짜별 적용 검증 완료
- 평시 96일: Stage 2 조정 없이 Stage 1 유지
- 이벤트 26일: 이벤트 종류별 중앙값 조정 적용
- fallback·복수 이벤트 적용: 0일 / 0일
- 이벤트 날짜 평균 조정: -1,795.6통
- Test 실제 물량과의 성능 비교: 수행하지 않음


## 17. Test 전체·평시·이벤트 성능 최종 평가

앞 단계에서 확정한 Stage 1 모델과 Stage 2 조정값을 변경하지 않고 2026년 Test 실제 물량과 처음 비교한다. Test 결과를 본 뒤 후보를 다시 선택하거나 파라미터를 조정하지 않는다.

- 평가 구간: 전체 122일, 평시 96일, 이벤트 26일
- 비교 모델: Stage 1 단독 vs Stage 1 + Stage 2 하이브리드
- 1순위 판단 기준: 이벤트 구간 MAE
- 보조 지표: RMSE, WAPE, MAPE
- MAPE는 실제 물량이 매우 작은 날짜에서 과도하게 커질 수 있으므로 해석에 주의한다.

In [17]:
test_evaluation = test_hybrid_predictions.copy()
test_evaluation[TARGET_COLUMN] = test[TARGET_COLUMN].to_numpy()

assert len(test_evaluation) == len(test) == 122
assert test_evaluation["접수일자"].equals(test["접수일자"])
assert test_evaluation[TARGET_COLUMN].equals(test[TARGET_COLUMN])
assert test_evaluation[TARGET_COLUMN].gt(0).all()
assert np.isfinite(
    test_evaluation[
        [
            TARGET_COLUMN,
            "stage1_baseline_prediction",
            "hybrid_prediction",
        ]
    ]
).all(axis=None)

TEST_SEGMENTS = {
    "전체": pd.Series(True, index=test_evaluation.index),
    "평시": test_evaluation["is_event"].eq(0),
    "이벤트": test_evaluation["is_event"].eq(1),
}
TEST_MODEL_PREDICTIONS = {
    "Stage 1 단독": "stage1_baseline_prediction",
    "하이브리드": "hybrid_prediction",
}

test_metric_rows = []
for segment_name, segment_mask in TEST_SEGMENTS.items():
    segment = test_evaluation.loc[segment_mask]
    for model_name, prediction_column in TEST_MODEL_PREDICTIONS.items():
        test_metric_rows.append(
            {
                "구간": segment_name,
                "모델": model_name,
                "행 수": len(segment),
                **regression_metrics(
                    segment[TARGET_COLUMN],
                    segment[prediction_column],
                ),
            }
        )

test_metrics = pd.DataFrame(test_metric_rows)

stage1_test_metrics = test_metrics.loc[
    test_metrics["모델"].eq("Stage 1 단독")
].drop(columns="모델")
hybrid_test_metrics = test_metrics.loc[
    test_metrics["모델"].eq("하이브리드")
].drop(columns="모델")

test_model_comparison = stage1_test_metrics.merge(
    hybrid_test_metrics,
    on=["구간", "행 수"],
    suffixes=("_Stage1", "_하이브리드"),
)
for metric in ["MAE", "RMSE", "MAPE(%)", "WAPE(%)"]:
    test_model_comparison[f"{metric}_개선율(%)"] = (
        (
            test_model_comparison[f"{metric}_Stage1"]
            - test_model_comparison[f"{metric}_하이브리드"]
        )
        / test_model_comparison[f"{metric}_Stage1"]
        * 100
    )

event_test_comparison = test_model_comparison.loc[
    test_model_comparison["구간"].eq("이벤트")
].iloc[0]
EVENT_MAE_SUCCESS = bool(
    event_test_comparison["MAE_하이브리드"]
    < event_test_comparison["MAE_Stage1"]
)

event_type_metric_rows = []
for event_column in test_active_event_types:
    event_type_data = test_evaluation.loc[
        test_evaluation[event_column].eq(1)
    ]
    for model_name, prediction_column in TEST_MODEL_PREDICTIONS.items():
        event_type_metric_rows.append(
            {
                "이벤트": event_column,
                "모델": model_name,
                "행 수": len(event_type_data),
                **regression_metrics(
                    event_type_data[TARGET_COLUMN],
                    event_type_data[prediction_column],
                ),
            }
        )

event_type_metrics = pd.DataFrame(event_type_metric_rows)
event_type_stage1 = event_type_metrics.loc[
    event_type_metrics["모델"].eq("Stage 1 단독")
].drop(columns="모델")
event_type_hybrid = event_type_metrics.loc[
    event_type_metrics["모델"].eq("하이브리드")
].drop(columns="모델")
event_type_comparison = event_type_stage1.merge(
    event_type_hybrid,
    on=["이벤트", "행 수"],
    suffixes=("_Stage1", "_하이브리드"),
)
event_type_comparison["MAE_개선율(%)"] = (
    (
        event_type_comparison["MAE_Stage1"]
        - event_type_comparison["MAE_하이브리드"]
    )
    / event_type_comparison["MAE_Stage1"]
    * 100
)

event_daily_evaluation = test_evaluation.loc[
    TEST_SEGMENTS["이벤트"],
    [
        "접수일자",
        "active_event_types",
        TARGET_COLUMN,
        "stage1_baseline_prediction",
        "stage2_adjustment",
        "hybrid_prediction",
    ],
].copy()
event_daily_evaluation["Stage1_절대오차"] = np.abs(
    event_daily_evaluation[TARGET_COLUMN]
    - event_daily_evaluation["stage1_baseline_prediction"]
)
event_daily_evaluation["하이브리드_절대오차"] = np.abs(
    event_daily_evaluation[TARGET_COLUMN]
    - event_daily_evaluation["hybrid_prediction"]
)
event_daily_evaluation["절대오차_개선량"] = (
    event_daily_evaluation["Stage1_절대오차"]
    - event_daily_evaluation["하이브리드_절대오차"]
)

lowest_actual_test_rows = test_evaluation.nsmallest(
    5,
    TARGET_COLUMN,
)[
    [
        "접수일자",
        TARGET_COLUMN,
        "is_event",
        "active_event_types",
    ]
]
mape_is_distorted = bool(
    lowest_actual_test_rows[TARGET_COLUMN].min() < 1_000
)

worst_hybrid_test_rows = test_evaluation.assign(
    하이브리드_절대오차=np.abs(
        test_evaluation[TARGET_COLUMN]
        - test_evaluation["hybrid_prediction"]
    )
).nlargest(10, "하이브리드_절대오차")[
    [
        "접수일자",
        "active_event_types",
        TARGET_COLUMN,
        "stage1_baseline_prediction",
        "stage2_adjustment",
        "hybrid_prediction",
        "하이브리드_절대오차",
    ]
]

final_test_evaluation_summary = pd.DataFrame(
    {
        "항목": [
            "Test 평가 행",
            "평시 평가 행",
            "이벤트 평가 행",
            "1순위 성공 기준",
            "성공 여부",
            "Test 확인 후 모델 재선정",
            "MAPE 왜곡 주의 필요",
        ],
        "값": [
            len(test_evaluation),
            int(TEST_SEGMENTS["평시"].sum()),
            int(TEST_SEGMENTS["이벤트"].sum()),
            "하이브리드 이벤트 MAE < Stage 1 이벤트 MAE",
            "충족" if EVENT_MAE_SUCCESS else "미충족",
            "수행하지 않음",
            "예" if mape_is_distorted else "아니오",
        ],
    }
)

display(final_test_evaluation_summary)
display(test_metrics)
display(test_model_comparison)
display(event_type_comparison)
display(event_daily_evaluation)
display(lowest_actual_test_rows)
display(worst_hybrid_test_rows)

print("2026년 Test 최종 평가 완료")
print(
    "- 이벤트 MAE | Stage 1: "
    f"{event_test_comparison['MAE_Stage1']:,.1f}, "
    "하이브리드: "
    f"{event_test_comparison['MAE_하이브리드']:,.1f}"
)
print(
    "- 이벤트 MAE 개선율: "
    f"{event_test_comparison['MAE_개선율(%)']:,.2f}%"
)
print(
    "- 사전 정의 성공 기준: "
    f"{'충족' if EVENT_MAE_SUCCESS else '미충족'}"
)
if mape_is_distorted:
    print(
        "- 주의: 실제 물량 1,000통 미만 행이 있어 "
        "MAPE보다 MAE·RMSE·WAPE를 우선 해석함"
    )
print("- Test 확인 후 모델 재선정·튜닝: 수행하지 않음")

,항목,값
0,Test 평가 행,122
1,평시 평가 행,96
2,이벤트 평가 행,26
3,1순위 성공 기준,하이브리드 이벤트 MAE < Stage 1 이벤트 MAE
4,성공 여부,미충족
5,Test 확인 후 모델 재선정,수행하지 않음
6,MAPE 왜곡 주의 필요,아니오


,구간,모델,행 수,MAE,RMSE,MAPE(%),WAPE(%)
0,전체,Stage 1 단독,122,"57,262.900","303,241.678",40.052,54.793
1,전체,하이브리드,122,"57,471.711","303,868.202",40.344,54.992
2,평시,Stage 1 단독,96,"24,825.536","34,340.612",37.332,33.902
3,평시,하이브리드,96,"24,825.536","34,340.612",37.332,33.902
4,이벤트,Stage 1 단독,26,"177,031.626","653,551.310",50.094,80.465
5,이벤트,하이브리드,26,"178,011.431","654,915.356",51.466,80.910


,구간,행 수,MAE_Stage1,RMSE_Stage1,MAPE(%)_Stage1,WAPE(%)_Stage1,MAE_하이브리드,RMSE_하이브리드,MAPE(%)_하이브리드,WAPE(%)_하이브리드,MAE_개선율(%),RMSE_개선율(%),MAPE(%)_개선율(%),WAPE(%)_개선율(%)
0,전체,122,"57,262.900","303,241.678",40.052,54.793,"57,471.711","303,868.202",40.344,54.992,-0.365,-0.207,-0.730,-0.365
1,평시,96,"24,825.536","34,340.612",37.332,33.902,"24,825.536","34,340.612",37.332,33.902,0.000,0.000,0.000,0.000
2,이벤트,26,"177,031.626","653,551.310",50.094,80.465,"178,011.431","654,915.356",51.466,80.910,-0.553,-0.209,-2.739,-0.553


,이벤트,행 수,MAE_Stage1,RMSE_Stage1,MAPE(%)_Stage1,WAPE(%)_Stage1,MAE_하이브리드,RMSE_하이브리드,MAPE(%)_하이브리드,WAPE(%)_하이브리드,MAE_개선율(%)
0,제1기분 자동차세,5,"67,366.447","85,629.633",71.241,58.713,"71,019.222","79,239.669",96.916,61.897,-5.422
1,사회보험료 통합,21,"203,142.383","726,003.411",45.059,82.890,"203,485.766","727,696.058",40.645,83.030,-0.169


,접수일자,active_event_types,접수통수,stage1_baseline_prediction,stage2_adjustment,hybrid_prediction,Stage1_절대오차,하이브리드_절대오차,절대오차_개선량
13,2026-01-21,사회보험료 통합,34562,"54,582.755","-6,571.633","48,011.122","20,020.755","13,449.122","6,571.633"
14,2026-01-22,사회보험료 통합,81196,"67,114.559","-6,571.633","60,542.927","14,081.441","20,653.073","-6,571.633"
15,2026-01-23,사회보험료 통합,39643,"61,128.892","-6,571.633","54,557.259","21,485.892","14,914.259","6,571.633"
33,2026-02-23,사회보험료 통합,137344,"104,224.463","-6,571.633","97,652.830","33,119.537","39,691.170","-6,571.633"
34,2026-02-24,사회보험료 통합,37136,"52,780.488","-6,571.633","46,208.855","15,644.488","9,072.855","6,571.633"
35,2026-02-25,사회보험료 통합,55869,"50,978.980","-6,571.633","44,407.347","4,890.020","11,461.653","-6,571.633"
52,2026-03-23,사회보험료 통합,113857,"95,187.789","-6,571.633","88,616.157","18,669.211","25,240.843","-6,571.633"
53,2026-03-24,사회보험료 통합,60034,"51,957.677","-6,571.633","45,386.044","8,076.323","14,647.956","-6,571.633"
54,2026-03-25,사회보험료 통합,108513,"60,704.223","-6,571.633","54,132.590","47,808.777","54,380.410","-6,571.633"
73,2026-04-21,사회보험료 통합,56086,"50,641.758","-6,571.633","44,070.125","5,444.242","12,015.875","-6,571.633"


,접수일자,접수통수,is_event,active_event_types
92,2026-05-20,16858,0,없음
114,2026-06-19,21217,0,없음
106,2026-06-09,22203,1,제1기분 자동차세
117,2026-06-24,22315,1,사회보험료 통합
98,2026-05-27,22808,0,없음


,접수일자,active_event_types,접수통수,stage1_baseline_prediction,stage2_adjustment,hybrid_prediction,하이브리드_절대오차
94,2026-05-22,사회보험료 통합,3325214,"48,256.573","-6,571.633","41,684.940","3,283,529.060"
95,2026-05-23,사회보험료 통합,595408,"33,941.965","-6,571.633","27,370.332","568,037.668"
6,2026-01-12,없음,277069,"132,861.479",0.000,"132,861.479","144,207.521"
109,2026-06-12,제1기분 자동차세,200192,"62,049.831","18,263.875","80,313.705","119,878.295"
22,2026-02-03,없음,170212,"58,044.677",0.000,"58,044.677","112,167.323"
108,2026-06-11,제1기분 자동차세,195110,"69,902.875","18,263.875","88,166.749","106,943.251"
7,2026-01-13,없음,169219,"76,313.702",0.000,"76,313.702","92,905.298"
96,2026-05-24,사회보험료 통합,109723,"31,230.475","-6,571.633","24,658.843","85,064.157"
82,2026-05-06,없음,55552,"133,186.365",0.000,"133,186.365","77,634.365"
84,2026-05-08,없음,138147,"63,791.334",0.000,"63,791.334","74,355.666"


2026년 Test 최종 평가 완료
- 이벤트 MAE | Stage 1: 177,031.6, 하이브리드: 178,011.4
- 이벤트 MAE 개선율: -0.55%
- 사전 정의 성공 기준: 미충족
- Test 확인 후 모델 재선정·튜닝: 수행하지 않음
